# StatCore linear models: calibrated, regularised and robust logistic / ridge regression

[Open in Colab](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/statcore-linear.ipynb)

**Paper topic:** Regularized, Calibrated, and Robust Linear Models in StatCore — `docs/latex/statcore-linear.tex` ([repository](https://github.com/Jabualani/ML-Reliabilty/blob/main/docs/latex/statcore-linear.tex)).
**Experiment script reproduced here:** `docs/latex/experiments/linear_experiment.py` (reduced seed count so the notebook runs in a few minutes).

StatCore is a pure-NumPy/SciPy model family in `ml_reliability.statcore`. Every model has a `variant`
argument; each variant below is compared with the standard reference implementation.
**scikit-learn is used only as the baseline/reference** (and for loading public datasets), never inside StatCore.

## 1. Install (parameter cell)

> **Note:** StatCore has **not been pushed / published yet**. Until the repository is public, the
> `git+https` install below will fail. In that case use the **fallback cell**: zip the repository folder,
> upload the zip to Colab (left sidebar → Files, or the `files.upload()` helper) and `pip install` it.
> Run exactly one of the two cells.

In [ ]:
# @title Install StatCore (PARAMETER: change the source if needed)
# Option A - once the repository is published:
!pip install "git+https://github.com/Jabualani/ML-Reliabilty.git"

In [ ]:
# @title FALLBACK install from an uploaded zip of the repository (use if Option A fails)
# Zip the repo folder (ML-Reliabilty.zip), then uncomment and run:
# from google.colab import files
# files.upload()                       # pick ML-Reliabilty.zip
# !unzip -q -o ML-Reliabilty.zip -d repo
# !pip install ./repo/ML-Reliabilty     # adjust to the folder that contains pyproject.toml

In [ ]:
# @title Parameters and imports
SEEDS = (11, 23, 37)   # the paper uses more seeds; fewer here to keep the run short

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 2. Method summary

Features are standardised with training statistics, $z=(x-m)/s$. Notation: $\ell_i=-\log p_{y_i}(z_i)$.

### Multinomial logistic regression
Softmax model $p(c\mid z)=\mathrm{softmax}(Wz+b)_c$, trained with L-BFGS on the weighted mean loss with L2 penalty on $W$ only:
$$\min_{W,b}\ \sum_i \tilde w_i\,\ell_i+\tfrac{\alpha}{2}\|W\|_F^2,\qquad \tilde w_i=w_i/\textstyle\sum w .$$
Reference: sklearn `Pipeline(StandardScaler, LogisticRegression(C=1/(n\alpha)))` (same penalty scale).

- **`base`** — fixed $\alpha$ (default $10^{-2}$).
- **`risk_calibrated`** — fit on a stratified split (`calib_fraction`=25 % held out), then a scalar temperature $T=e^{\tau}$, $\tau\in[-3,3]$, minimises the held-out NLL of $\mathrm{softmax}(\text{logits}/T)$ (golden-section search); `decision_function` divides logits by $T$.
- **`stat_regularised`** — stratified $K$-fold CV log-loss over $\alpha\in\{10^{-4},..,10\}$ (6 log-spaced values), refit on all data with the best $\alpha$ (`cv_scores_`).
- **`robust_loss`** — 6 reweighting rounds: $\mathrm{cut}=\mathrm{median}(\ell)+1.345\cdot1.4826\,\mathrm{MAD}(\ell)$, $w_i=\min(1,\mathrm{cut}/\ell_i)$ (Huber-type bounded influence for high-loss, e.g. mislabelled, points).

### Ridge regression
Closed form with unpenalised intercept: $\beta=(Z_a^\top WZ_a+\lambda P)^{-1}Z_a^\top Wy$, $P=\mathrm{diag}(0,1,..,1)$. Reference: sklearn `Pipeline(StandardScaler, Ridge(alpha=1))`.

- **`base`** — fixed $\lambda=\alpha$.
- **`risk_calibrated`** — fit on the training part of a split; linear recalibration $\hat y\leftarrow a+b\,\hat y$ fitted on the held-out part ($b=\mathrm{cov}(\hat y,y)/\mathrm{var}(\hat y)$); absolute held-out residuals are stored and `predict_interval(X, level)` returns the split-conformal band $\hat y\pm q$ with $q$ the $\lceil(m+1)\,\text{level}\rceil$-th smallest residual.
- **`stat_regularised`** — generalised cross-validation, with singular values $s_j$ of the centred design: $\mathrm{shrink}_j=\frac{s_j^2}{s_j^2+\lambda}$, $\mathrm{dof}=1+\sum_j\mathrm{shrink}_j$,
$$\mathrm{GCV}(\lambda)=\frac{n\,\mathrm{RSS}(\lambda)}{(n-\mathrm{dof})^2},$$
minimised on a 46-point log grid in $[10^{-6},10^{3}]$.
- **`robust_loss`** — Huber IRLS (up to 30 iterations): $\sigma=1.4826\,\mathrm{MAD}(r)$, $c=\kappa\sigma$ ($\kappa=1.345$), $w_i=\min(1,c/|r_i|)$.

## 3. Experiments (mirror `linear_experiment.py`)

Classification cases: `clean`, covariate `shifted` test set, 15 % `label_noise`, 8 % gross `outliers`, `high_dim` ($d=80$, $n=140$), and real `breast_cancer`. Regression cases: `clean`, `shifted`, `noisy` (extra training noise), `outliers` (8 % of $y$ corrupted), `high_dim`, and `diabetes`. Metrics: accuracy / log-loss / 10-bin ECE and MSE / MAE / $R^2$.

In [ ]:
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression      # baseline only
from sklearn.linear_model import Ridge as SklearnRidge                                # baseline only
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from ml_reliability.statcore import LogisticRegression, RidgeRegression
from ml_reliability.statcore.linear.linear import VARIANTS

CLASS_CASES = ("clean", "shifted", "label_noise", "outliers", "high_dim", "breast_cancer")
REG_CASES = ("clean", "shifted", "noisy", "outliers", "high_dim", "diabetes")

def split_indices(n, seed):
    rng = np.random.default_rng(seed); idx = rng.permutation(n); k = int(round(0.25 * n))
    return np.sort(idx[k:]), np.sort(idx[:k])

def class_data(case, seed):
    rng = np.random.default_rng(seed)
    if case == "breast_cancer":
        ds = load_breast_cancer(); tr, te = split_indices(len(ds.target), seed)
        return ds.data[tr], ds.target[tr], ds.data[te], ds.target[te]
    ntr, nte = (140, 300) if case == "high_dim" else (360, 600)
    d = 80 if case == "high_dim" else 12
    info = 10 if case == "high_dim" else min(d, 8)
    beta = np.zeros(d); beta[:info] = np.array([1.2, -1.0, 0.8, -0.7, 0.5, -0.4, 0.3, -0.2, 0.15, -0.1])[:info]
    shift = np.zeros(d)
    if case == "shifted":
        shift[:info] = np.linspace(0.35, 0.9, info)
    def sample(n, offset):
        x = rng.normal(size=(n, d)) + offset
        p = 1.0 / (1.0 + np.exp(-np.clip(x @ beta, -30, 30)))
        return x, rng.binomial(1, p)
    xtr, ytr = sample(ntr, np.zeros(d)); xte, yte = sample(nte, shift)
    if case == "label_noise":
        flip = rng.random(ntr) < 0.15; ytr[flip] = 1 - ytr[flip]
    elif case == "outliers":
        out = rng.random(ntr) < 0.08; xtr[out] = rng.normal(0, 6, size=(out.sum(), d))
    return xtr, ytr, xte, yte

def regression_data(case, seed):
    rng = np.random.default_rng(seed)
    if case == "diabetes":
        ds = load_diabetes(); tr, te = split_indices(len(ds.target), seed)
        return ds.data[tr], ds.target[tr], ds.data[te], ds.target[te]
    ntr, nte = (100, 300) if case == "high_dim" else (360, 600)
    d = 80 if case == "high_dim" else 12
    info = 10 if case == "high_dim" else min(d, 8)
    beta = np.zeros(d); beta[:info] = np.linspace(1.5, -0.7, info)
    offset = np.zeros(d)
    if case == "shifted":
        offset[:info] = np.linspace(0.4, 0.9, info)
    xtr = rng.normal(size=(ntr, d)); xte = rng.normal(size=(nte, d)) + offset
    ytr = xtr @ beta + rng.normal(0, 1.0, ntr); yte = xte @ beta + rng.normal(0, 1.0, nte)
    if case == "noisy":
        ytr += rng.normal(0, 3.0, ntr)
    elif case == "outliers":
        out = rng.random(ntr) < 0.08; ytr[out] += rng.normal(0, 20.0, out.sum())
    return xtr, ytr, xte, yte

def class_scores(y, p):
    pred = p.argmax(axis=1); acc = float(np.mean(pred == y))
    loss = float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-15, 1)).mean())
    conf, correct = p.max(axis=1), pred == y; edges = np.linspace(0, 1, 11); ece = 0.0
    for j in range(10):
        m = (conf >= edges[j]) & (conf < edges[j + 1] if j < 9 else conf <= edges[j + 1])
        if m.any(): ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return dict(accuracy=acc, logloss=loss, ece=float(ece))

def reg_scores(y, pred):
    err = pred - y
    return dict(mse=float(np.mean(err ** 2)), mae=float(np.mean(np.abs(err))),
                r2=float(1.0 - np.sum(err ** 2) / np.sum((y - y.mean()) ** 2)))

crow, rrow = [], []
for case in CLASS_CASES:
    for seed in SEEDS:
        xtr, ytr, xte, yte = class_data(case, seed)
        base = make_pipeline(StandardScaler(), SklearnLogisticRegression(C=1.0 / (len(ytr) * 1e-2), max_iter=1000, random_state=seed)).fit(xtr, ytr)
        crow.append(dict(case=case, method="sklearn_standard", seed=seed, **class_scores(yte, base.predict_proba(xte))))
        for v in VARIANTS:
            with np.errstate(divide="ignore", invalid="ignore"):
                m = LogisticRegression(variant=v, random_state=seed).fit(xtr, ytr)
            crow.append(dict(case=case, method=v, seed=seed, **class_scores(yte, m.predict_proba(xte))))
for case in REG_CASES:
    for seed in SEEDS:
        xtr, ytr, xte, yte = regression_data(case, seed)
        base = make_pipeline(StandardScaler(), SklearnRidge(alpha=1.0)).fit(xtr, ytr)
        rrow.append(dict(case=case, method="sklearn_standard", seed=seed, **reg_scores(yte, base.predict(xte))))
        for v in VARIANTS:
            m = RidgeRegression(variant=v, random_state=seed).fit(xtr, ytr)
            rrow.append(dict(case=case, method=v, seed=seed, **reg_scores(yte, m.predict(xte))))
cres, rres = pd.DataFrame(crow), pd.DataFrame(rrow)
METHODS = ["sklearn_standard", *VARIANTS]
def pivot(df, metric, cases):
    return df.groupby(["case", "method"])[metric].mean().unstack("method").loc[list(cases), METHODS].round(4)
print(f"CLASSIFICATION, mean over seeds {SEEDS}")
for metric in ("accuracy", "logloss", "ece"):
    print(f"\n[{metric}]"); print(pivot(cres, metric, CLASS_CASES).to_string())

In [ ]:
print("REGRESSION")
for metric in ("mse", "mae", "r2"):
    print(f"\n[{metric}]"); print(pivot(rres, metric, REG_CASES).to_string())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
pivot(cres, "logloss", CLASS_CASES).plot.bar(ax=axes[0], rot=30, legend=False); axes[0].set_title("classification log-loss"); axes[0].set_yscale("log")
pivot(cres, "ece", CLASS_CASES).plot.bar(ax=axes[1], rot=30, legend=False); axes[1].set_title("classification ECE")
pivot(rres, "mse", REG_CASES).plot.bar(ax=axes[2], rot=30); axes[2].set_title("regression MSE"); axes[2].set_yscale("log")
axes[2].legend(fontsize=7); plt.tight_layout(); plt.show()

## 4. Look inside each variant

In [ ]:
seed = SEEDS[0]
xtr, ytr, xte, yte = class_data("label_noise", seed)
rc = LogisticRegression(variant="risk_calibrated", random_state=seed).fit(xtr, ytr)
sr = LogisticRegression(variant="stat_regularised", random_state=seed).fit(xtr, ytr)
rl = LogisticRegression(variant="robust_loss", random_state=seed).fit(xtr, ytr)
flipped_weight = rl.sample_weights_
print(f"risk_calibrated temperature T = {rc.temperature_:.3f}  (T>1 softens over-confident logits)")
print("stat_regularised CV log-loss by alpha:", {k: round(float(v), 4) for k, v in sr.cv_scores_.items()}, "-> alpha =", sr.alpha_)
print(f"robust_loss: {int((flipped_weight < 0.999).sum())} of {len(flipped_weight)} samples down-weighted; min weight {flipped_weight.min():.3f}")

# reliability diagram (label_noise): plain vs risk_calibrated
plain = LogisticRegression(variant="base", random_state=seed).fit(xtr, ytr)
plt.figure(figsize=(4.6, 4))
for name, mdl in (("base", plain), ("risk_calibrated", rc)):
    p = mdl.predict_proba(xte); conf = p.max(1); corr = p.argmax(1) == yte
    bins = np.linspace(0.5, 1, 7); idx = np.clip(np.digitize(conf, bins) - 1, 0, 5)
    xs = [conf[idx == b].mean() for b in range(6) if (idx == b).any()]; ys = [corr[idx == b].mean() for b in range(6) if (idx == b).any()]
    plt.plot(xs, ys, "o-", label=name)
plt.plot([0.5, 1], [0.5, 1], "k--"); plt.xlabel("confidence"); plt.ylabel("accuracy"); plt.legend(); plt.title("Reliability (label_noise)"); plt.show()

# Ridge: conformal interval, GCV alpha, Huber weights
xtr, ytr, xte, yte = regression_data("outliers", seed)
rcal = RidgeRegression(variant="risk_calibrated", random_state=seed).fit(xtr, ytr)
lo, hi = rcal.predict_interval(xte, level=0.9)
print(f"split-conformal 90% interval: coverage {np.mean((yte >= lo) & (yte <= hi)):.3f}, mean width {np.mean(hi - lo):.2f}  (calibration a,b = {np.round(rcal.calib_, 3)})")
print("GCV-selected ridge alpha:", RidgeRegression(variant="stat_regularised").fit(xtr, ytr).alpha_)
rrob = RidgeRegression(variant="robust_loss").fit(xtr, ytr)
print(f"Huber IRLS: {int((rrob.sample_weights_ < 0.999).sum())} of {len(ytr)} samples down-weighted (true outliers ~ 8%)")

## 5. Summary and limitations

In [ ]:
def best(df, metric, cases, smaller=True):
    p = pivot(df, metric, cases); return (p.idxmin(axis=1) if smaller else p.idxmax(axis=1)).to_dict()
print("best log-loss per classification case:", best(cres, "logloss", CLASS_CASES))
print("best accuracy per classification case:", best(cres, "accuracy", CLASS_CASES, smaller=False))
print("best MSE per regression case:", best(rres, "mse", REG_CASES))

**What to read from the tables.** `base` tracks the sklearn pipeline (same objective; small differences from solver details). In our local 3-seed run: `robust_loss` had the best log-loss/ECE under `label_noise` and `outliers` and the best accuracy there, and for ridge cut MSE on `outliers` by ~50 % (≈2.06 → 1.06); `stat_regularised` helped most in `high_dim` for both models (ridge MSE 4.34 → 2.61); `risk_calibrated` reduced ECE only in `high_dim` and was *worse* than `base` on most other cases (e.g. breast-cancer log-loss 0.37 vs 0.11).

**Honest limitations**
- `risk_calibrated` trains on only 75 % of the data and its temperature is fit on a small held-out set, so on small or already well-calibrated problems it can hurt (see breast cancer); a single scalar temperature cannot fix mis-specification or covariate shift. Ridge `risk_calibrated` likewise lost accuracy everywhere except as an interval generator.
- `robust_loss` (logistic) raised ECE/log-loss on the clean cases (it down-weights hard but legitimate points) and lowered breast-cancer accuracy.
- GCV selected $\lambda=100$ on the outlier case (the grid spans $10^{-6}$–$10^{3}$), where GCV itself is distorted by the outliers.
- Mostly synthetic data with known structure and a few seeds; real datasets are small (breast cancer 569, diabetes 442). Gains of the order of one SD are not evidence.
- The conformal interval gives marginal (average) coverage under exchangeability only; it is not valid under covariate shift and exists only for ridge `risk_calibrated`. CV/GCV grids are fixed and coarse and ignore distribution shift.